# StemTOCvitro

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.StemTOCvitro)

class StemTOCvitro(stemTOC):
    pass



In [3]:
model = pya.models.StemTOCvitro()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "stemtocvitro"
model.metadata["data_type"] = "DNA methylation"  # Paper: The study constructs clocks from DNA methylation measurements.
model.metadata["species"] = "Homo sapiens"  # Paper: The analyzed samples and clock are human.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Zhu, Tianlei, et al. \"An improved epigenetic counter to track mitotic age in cells.\" Nature Communications 15 (2024): 4211."
model.metadata["doi"] = "https://doi.org/10.1038/s41467-024-48649-8"
model.metadata["notes"] = "In-vitro precursor of stemTOC based on the 95th percentile across 629 population-doubling-associated CpGs."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["multi-tissue", "cultured human cells"]  # Paper: Feature selection used fetal/neonatal reference tissues and cultured normal cell types; stemTOC also used blood cohorts.
model.metadata["predicts"] = ["mitotic age"]  # Paper: The paper describes the returned score as relative mitotic age.
model.metadata["training_target"] = ["population doublings"]  # Paper: Candidate CpGs were selected for methylation gain with population doublings and, where applicable, age in blood.
model.metadata["unit"] = ["beta value"]  # Paper: The score is the 95% upper quantile of CpG DNAm beta values.
model.metadata["model_type"] = "95th-percentile methylation aggregation"  # Paper: The score is computed as the 95% upper quantile across selected CpGs.
model.metadata["platform"] = ["Illumina 450K", "Illumina EPIC"]  # Paper: Derivation datasets included 450K fetal samples and EPIC cultured-cell samples.
model.metadata["population"] = "prenatal and newborn"  # Paper: The derivation cohorts are fetal/neonatal tissues, normal cell cultures, and for stemTOC adult blood cohorts.
model.metadata["journal"] = "Nature Communications"
model.metadata["last_author"] = "Andrew E. Teschendorff"
model.metadata["n_features"] = 629
model.metadata["citations"] = 24
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/StemTOCvitro.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
if str(df.columns[0]).startswith('Unnamed'):
    df = df.iloc[:, 1:]
model.features = df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor([1.0]).unsqueeze(0)
intercept = torch.tensor([0.0])

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = [-1]*len(model.features)

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "0.95 quantile"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Zhu, Tianyu, et al. "A pan-tissue DNA methylation atlas enables '
             'in silico decomposition of human tissue methylomes at cell-type '
             'resolution." Nature Communications 15 (2024).',
 'clock_name': 'stemtocvitro',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/s41467-024-48649-8',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2024}
reference_values: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]... [Total elements: 629]
preprocess_name: '0.95 quantile'
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg20327258', 'cg23062112', 'cg07365816', 'cg08659394', 'cg14575559', 'cg13294856', 'cg18560328', 'cg2122926

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
